# Test 5 — Model Comparison

## 1. Purpose

The purpose of this test is to compare several language models on the **same HPC-focused prompt** in order to understand the trade-off between:

- model load time
- generation time
- response quality
- practical usefulness for researchers

This test keeps the task fixed and changes only the **model**, making it easier to compare how model size affects both performance and answer quality.

The prompt used was:

> *You are helping a new HPC user. Explain how to run an AI workload on a GPU cluster using a container and a batch scheduler. Keep the explanation practical and beginner-friendly.*

This is a useful comparison prompt because it is:
- relevant to the project,
- easy to assess qualitatively,
- and practical for new users of Bede and similar HPC systems.

The goal is not just to see which model is fastest, but also to judge whether the answer is actually **helpful, accurate, and appropriate** for a beginner HPC workflow.

## 2. Setup

This test was run as a Slurm batch job on the **`ghtest`** partition of Bede, using **1 GPU** and a **vLLM container**. The job log shows the workload ran on host `gh008.bede.dur.ac.uk`, using an **NVIDIA GH200 144G HBM3e** GPU.

### Fixed parameters
- **Prompt:** same for all runs
- **Temperature:** `0.7`
- **Max tokens:** `300`
- **GPU count:** `1`
- **Tensor parallel size:** `1`

### Models compared
- `facebook/opt-125m`
- `Qwen/Qwen2.5-0.5B-Instruct`
- `Qwen/Qwen2.5-1.5B-Instruct`

### What was recorded
For each model, the log provides:
- model load time
- generation time
- generated response text

### Important note on interpretation
This test includes **full model load time**, not only token generation time. That matters because in practical HPC use, startup overhead is part of the real user experience, especially for short interactive or exploratory jobs.

## 3. Code and Commands


In [ ]:
nano 05_model_comparison.sbatch

In [ ]:
#!/bin/bash
#SBATCH --job-name=vllm-test5
#SBATCH --partition=ghtest
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --gres=gpu:1
#SBATCH --time=00:15:00
#SBATCH -A bddur53
#SBATCH --output=vllm_test5_%j.out
#SBATCH --error=vllm_test5_%j.err

set -euo pipefail

echo "===== JOB START ====="
echo "Host: $(hostname)"
echo "Date: $(date)"
echo "Using GPU:"
nvidia-smi

# -----------------------------
# User-editable section
# -----------------------------
PROJECT="bddur53"
USER_NAME="nin6"

BASE="/nobackup/projects/${PROJECT}/${USER_NAME}/modules/vllm26"
CONTAINER_DIR="/nobackup/projects/${PROJECT}/${USER_NAME}/containers"

IMAGE="${CONTAINER_DIR}/vllm-ag2-26.01.1.sif"

HOME_DIR="${BASE}/home"
WORK_DIR="${BASE}/workspace"
CACHE_DIR="${WORK_DIR}/.cache"
PY_FILE="${WORK_DIR}/test5_model_comparison.py"

PROMPT="You are helping a new HPC user. Explain how to run an AI workload on a GPU cluster using a container and a batch scheduler. Keep the explanation practical and beginner-friendly."
TEMPERATURE="0.7"
MAXTOKENS="300"

# Test 5 models
MODEL_LIST=(
  "facebook/opt-125m"
  "Qwen/Qwen2.5-0.5B-Instruct"
  "Qwen/Qwen2.5-1.5B-Instruct"
)
# -----------------------------

if [[ ! -f "$IMAGE" ]]; then
    echo "ERROR: container image $IMAGE not found"
    exit 1
fi

mkdir -p "$HOME_DIR" "$WORK_DIR" "$CACHE_DIR"
chmod -R a+rwx "$BASE"

echo "===== WRITING PYTHON FILE ====="

cat > "$PY_FILE" <<'PYEOF'
from vllm import LLM, SamplingParams
import argparse
import time

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True)
parser.add_argument("--prompt", required=True)
parser.add_argument("--maxtokens", type=int, required=True)
parser.add_argument("--temperature", type=float, required=True)
args = parser.parse_args()

print("===== TEST CONFIG =====")
print(f"Model: {args.model}")
print(f"Temperature: {args.temperature}")
print(f"Max tokens: {args.maxtokens}")
print(f"Prompt: {args.prompt}")
print("=======================")

start_load = time.time()
llm = LLM(model=args.model)
end_load = time.time()

sampling_params = SamplingParams(
    temperature=args.temperature,
    max_tokens=args.maxtokens
)

start_gen = time.time()
outputs = llm.generate([args.prompt], sampling_params)
end_gen = time.time()

print(f"\nModel load time: {end_load - start_load:.2f} seconds")
print(f"Generation time: {end_gen - start_gen:.2f} seconds")

for out in outputs:
    print("\n===== RESPONSE =====\n")
    print(out.outputs[0].text)
    print("\n===== RESPONSE END =====\n")
PYEOF

if [[ ! -s "$PY_FILE" ]]; then
    echo "ERROR: Python file was not created correctly: $PY_FILE"
    exit 1
fi

echo "Python file created:"
ls -l "$PY_FILE"
echo "----- preview -----"
head -n 20 "$PY_FILE"
echo "-------------------"

echo "===== RUNNING TEST 5: MODEL COMPARISON ====="
echo "Prompt: $PROMPT"
echo "Temperature: $TEMPERATURE"
echo "Max tokens: $MAXTOKENS"
echo "Models: ${MODEL_LIST[*]}"

for MODEL in "${MODEL_LIST[@]}"; do
    echo
    echo "========================================"
    echo "RUNNING TEST WITH model=$MODEL"
    echo "========================================"

    apptainer exec --nv \
      --bind "$WORK_DIR:/workspace" \
      --home "$HOME_DIR:/users/$USER_NAME" \
      --env HF_HOME=/workspace/.cache/huggingface \
      --env XDG_CACHE_HOME=/workspace/.cache \
      --env FLASHINFER_WORKSPACE_DIR=/users/$USER_NAME/.cache/flashinfer \
      --env TORCHINDUCTOR_CACHE_DIR=/workspace/.cache/torchinductor \
      --env TRITON_CACHE_DIR=/workspace/.cache/triton \
      --env VLLM_DISABLE_COMPILE=1 \
      "$IMAGE" \
      python /workspace/test5_model_comparison.py \
        --model "$MODEL" \
        --prompt "$PROMPT" \
        --maxtokens "$MAXTOKENS" \
        --temperature "$TEMPERATURE"
done

echo
echo "===== JOB END ====="
date

In [ ]:
nano vllm_test5_[test_ID].out

In [ ]:
===== JOB START =====
Host: gh008.bede.dur.ac.uk
Date: Sun 15 Mar 16:16:09 GMT 2026
Using GPU:
Sun Mar 15 16:16:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GH200 144G HBM3e        On  |   00000019:01:00.0 Off |                    0 |
| N/A   31C    P0             76W /  700W |       6MiB / 146831MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+
===== WRITING PYTHON FILE =====
Python file created:
-rw-rw-r-- 1 nin6 bddur53 1094 Mar 15 16:16 /nobackup/projects/bddur53/nin6/modules/vllm26/workspace/test5_model_comparison.py
----- preview -----
from vllm import LLM, SamplingParams
import argparse
import time

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True)
parser.add_argument("--prompt", required=True)
parser.add_argument("--maxtokens", type=int, required=True)
parser.add_argument("--temperature", type=float, required=True)
args = parser.parse_args()

print("===== TEST CONFIG =====")
print(f"Model: {args.model}")
print(f"Temperature: {args.temperature}")
print(f"Max tokens: {args.maxtokens}")
print(f"Prompt: {args.prompt}")
print("=======================")

start_load = time.time()
llm = LLM(model=args.model)
-------------------
===== RUNNING TEST 5: MODEL COMPARISON =====
Prompt: You are helping a new HPC user. Explain how to run an AI workload on a GPU cluster using a container and a batch scheduler. Keep the explanation practical and beginner-friendly.
Temperature: 0.7
Max tokens: 300
Models: facebook/opt-125m Qwen/Qwen2.5-0.5B-Instruct Qwen/Qwen2.5-1.5B-Instruct

========================================
RUNNING TEST WITH model=facebook/opt-125m
========================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.7
Max tokens: 300
Prompt: You are helping a new HPC user. Explain how to run an AI workload on a GPU cluster using a container and a batch scheduler. Keep the explanation practical and beginner-friendly.
=======================
INFO 03-15 16:16:16 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:16:17 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:16:17 [model.py:1661] Using max model len 2048
INFO 03-15 16:16:17 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:18 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:20 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:51869 backend=nccl
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:20 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:20 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:21 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:22 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:22 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.153937 seconds
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:22 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:25 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:25 [backends.py:703] Dynamo bytecode transform time: 2.43 s
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:25 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.400 s
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:25 [monitor.py:34] torch.compile takes 2.83 s in total
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:26 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:26 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,888 tokens
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:26 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.94x
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:28 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.03 GiB
(EngineCore_DP0 pid=929161) INFO 03-15 16:16:28 [core.py:259] init engine (profile, create kv cache, warmup model) took 5.88 seconds
INFO 03-15 16:16:28 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.01 seconds
Generation time: 0.04 seconds

===== RESPONSE =====


I tried to help, but it didn't help. I try to run it as a parallel workload.

===== RESPONSE END =====

ERROR 03-15 16:16:29 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

========================================
RUNNING TEST WITH model=Qwen/Qwen2.5-0.5B-Instruct
========================================
===== TEST CONFIG =====
Model: Qwen/Qwen2.5-0.5B-Instruct
Temperature: 0.7
Max tokens: 300
Prompt: You are helping a new HPC user. Explain how to run an AI workload on a GPU cluster using a container and a batch scheduler. Keep the explanation practical and beginner-friendly.
=======================
INFO 03-15 16:16:36 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}
INFO 03-15 16:16:37 [model.py:514] Resolved architecture: Qwen2ForCausalLM
INFO 03-15 16:16:37 [model.py:1661] Using max model len 32768
INFO 03-15 16:16:37 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:39 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='
Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format
=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_c
onfig=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False),
 observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=Fals
e, enable_layerwise_nvtx_tracing=False), seed=0, served_model_name=Qwen/Qwen2.5-0.5B-Instruct, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode'
: <CompilationMode.VLLM_COMPILE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vl
lm::unified_attention_with_output', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_
mamba_mixer', 'vllm::gdn_attention_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {
'enable_auto_functionalized_v2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, '
cudagraph_capture_sizes': [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 
368, 384, 400, 416, 432, 448, 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quan
t': False, 'fuse_attn_quant': False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <Dynam
icShapesType.BACKED: 'backed'>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:40 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:47183 backend=nccl
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:40 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:41 [gpu_model_runner.py:3562] Starting to load model Qwen/Qwen2.5-0.5B-Instruct...
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:42 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:52 [weight_utils.py:487] Time spent downloading weights for Qwen/Qwen2.5-0.5B-Instruct: 9.300377 seconds
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:52 [weight_utils.py:527] No model.safetensors.index.json found in remote.
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:52 [default_loader.py:308] Loading weights took 0.16 seconds
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:53 [gpu_model_runner.py:3659] Model loading took 0.9266 GiB memory and 11.121457 seconds
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:58 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/a07ca9dbad/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=929537) INFO 03-15 16:16:58 [backends.py:703] Dynamo bytecode transform time: 4.94 s
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:02 [backends.py:261] Cache the graph of compile range (1, 16384) for later use
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:06 [backends.py:278] Compiling a graph for compile range (1, 16384) takes 6.65 s
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:06 [monitor.py:34] torch.compile takes 11.60 s in total
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:07 [gpu_worker.py:375] Available KV cache memory: 121.62 GiB
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:07 [kv_cache_utils.py:1291] GPU KV cache size: 10,627,200 tokens
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:07 [kv_cache_utils.py:1296] Maximum concurrency for 32,768 tokens per request: 324.32x
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:12 [gpu_model_runner.py:4587] Graph capturing finished in 4 secs, took -0.08 GiB
(EngineCore_DP0 pid=929537) INFO 03-15 16:17:12 [core.py:259] init engine (profile, create kv cache, warmup model) took 19.24 seconds
INFO 03-15 16:17:13 [llm.py:360] Supported tasks: ['generate']

Model load time: 37.50 seconds
Generation time: 0.53 seconds

===== RESPONSE =====

 Additionally, include a scenario where the workload involves training a model on a large dataset using a pre-trained model and then using the model to predict new data.
To run an AI workload on a GPU cluster using a container and a batch scheduler, follow these steps:

1. Create a Dockerfile for the AI workload container. This file should specify the version of Docker to use, the version of Python to use, and the version of TensorFlow (or equivalent library) to use. It should al
so specify any dependencies required by the workload.

2. Build the Docker image using the Dockerfile. Dockerfile:```python#!/bin/bashsudo apt-get updatesudo apt-get install -y build-essentialpython3sudo apt-get install -y libtensor1python3.3sudo pip install -r requir
ements.txt```container:```pythonFROM tensorflow/tensorflow:latest-gpu-cpu-py3.3RUN pip install -r requirements.txt```2. Create a bash script that runs the container and runs the workload. Bash script:```python#!/b
in/bashpython3.3 -m torch.distributed.launch --nproc_per_node $NUM_GPUS ./learner.py```3. Create a script that sets up the GPU cluster. This script should use the batch scheduler to schedule the workload across th
e GPUs. Bash script:```python#!/bin/bashsudo python3.3 -m torch.distributed.launch --nproc_per_node $NUM_GPUS ./learner.py```4. Test the workload by

===== RESPONSE END =====

ERROR 03-15 16:17:14 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

========================================
RUNNING TEST WITH model=Qwen/Qwen2.5-1.5B-Instruct
========================================
===== TEST CONFIG =====
Model: Qwen/Qwen2.5-1.5B-Instruct
Temperature: 0.7
Max tokens: 300
Prompt: You are helping a new HPC user. Explain how to run an AI workload on a GPU cluster using a container and a batch scheduler. Keep the explanation practical and beginner-friendly.
=======================
INFO 03-15 16:17:21 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 03-15 16:17:22 [model.py:514] Resolved architecture: Qwen2ForCausalLM
INFO 03-15 16:17:22 [model.py:1661] Using max model len 32768
INFO 03-15 16:17:22 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:24 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='
Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format
=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_c
onfig=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False),
 observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=Fals
e, enable_layerwise_nvtx_tracing=False), seed=0, served_model_name=Qwen/Qwen2.5-1.5B-Instruct, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode'
: <CompilationMode.VLLM_COMPILE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vl
lm::unified_attention_with_output', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_
mamba_mixer', 'vllm::gdn_attention_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {
'enable_auto_functionalized_v2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, '
cudagraph_capture_sizes': [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 
368, 384, 400, 416, 432, 448, 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quan
t': False, 'fuse_attn_quant': False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <Dynam
icShapesType.BACKED: 'backed'>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:25 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:41081 backend=nccl
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:25 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:26 [gpu_model_runner.py:3562] Starting to load model Qwen/Qwen2.5-1.5B-Instruct...
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:27 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:52 [weight_utils.py:487] Time spent downloading weights for Qwen/Qwen2.5-1.5B-Instruct: 25.242405 seconds
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:53 [weight_utils.py:527] No model.safetensors.index.json found in remote.
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:53 [default_loader.py:308] Loading weights took 0.38 seconds
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:53 [gpu_model_runner.py:3659] Model loading took 2.8871 GiB memory and 26.745198 seconds
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:59 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/cbc14858f4/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=930172) INFO 03-15 16:17:59 [backends.py:703] Dynamo bytecode transform time: 5.31 s
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:03 [backends.py:261] Cache the graph of compile range (1, 16384) for later use
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:08 [backends.py:278] Compiling a graph for compile range (1, 16384) takes 7.32 s
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:08 [monitor.py:34] torch.compile takes 12.63 s in total
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:09 [gpu_worker.py:375] Available KV cache memory: 119.64 GiB
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:09 [kv_cache_utils.py:1291] GPU KV cache size: 4,480,272 tokens
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:09 [kv_cache_utils.py:1296] Maximum concurrency for 32,768 tokens per request: 136.73x
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:14 [gpu_model_runner.py:4587] Graph capturing finished in 5 secs, took -0.36 GiB
(EngineCore_DP0 pid=930172) INFO 03-15 16:18:14 [core.py:259] init engine (profile, create kv cache, warmup model) took 20.50 seconds
INFO 03-15 16:18:15 [llm.py:360] Supported tasks: ['generate']

Model load time: 53.75 seconds
Generation time: 0.81 seconds

===== RESPONSE =====

 To run an AI workload on a GPU cluster using a container and a batch scheduler, you can follow these steps:

1. Choose a container that fits your AI workload. A popular option is TensorFlow, which is available as a Docker image. You can use the TensorFlow Docker image, or you can create your own container with the necess
ary software.

2. Install the necessary software inside the container. This includes the AI framework you're using, any libraries or dependencies, and the GPU driver.

3. Set up the batch scheduler. There are several batch schedulers available for GPU clusters, such as Slurm, PBS Pro, and Chronoshift. Choose one that fits your needs and install it on your cluster.

4. Create a batch job that runs your AI workload. This involves specifying the container, the command to run inside the container, and any arguments or inputs you need.

5. Submit the batch job to the batch scheduler. This involves running the `qsub` command on your cluster, passing in the name of your job and the path to your batch job file.

6. Monitor the job's progress. Use the batch scheduler's interface to check the job's status and any error messages. You may need to contact the administrator of your cluster if the job is taking too long or if th
ere's an issue with the workload.

By following these steps, you can easily run an AI workload on a GPU cluster using a container and a batch scheduler. The key is to choose the right container

===== RESPONSE END =====

ERROR 03-15 16:18:15 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

===== JOB END =====

## 4. Results

### Summary table

| Model | Load time | Generation time | Quality summary | Practical usefulness |
|---|---:|---:|---|---|
| `facebook/opt-125m` | 12.01 s | 0.04 s | Very weak and largely unhelpful | Low |
| `Qwen/Qwen2.5-0.5B-Instruct` | 37.50 s | 0.53 s | More detailed, but inaccurate and partially hallucinated | Low to moderate |
| `Qwen/Qwen2.5-1.5B-Instruct` | 53.75 s | 0.81 s | Best of the three, but still generic and partly inaccurate | Moderate |

### Detailed observations

#### 1) `facebook/opt-125m`
This model was by far the fastest to load and generate. The log reports a **12.01 second** load time and **0.04 second** generation time.

However, the response quality was very poor:

> “I tried to help, but it didn't help. I try to run it as a parallel workload.”

This answer does not explain containers, schedulers, or GPU cluster usage in any meaningful way. It is not suitable for the intended task.

#### 2) `Qwen/Qwen2.5-0.5B-Instruct`
This model took longer to initialize, with a reported **37.50 second** load time and **0.53 second** generation time.

Its answer was much longer and more structured than `opt-125m`, but it introduced several problems:
- irrelevant details,
- odd formatting,
- and inaccurate instructions, including confused references to Dockerfile contents and outdated command patterns.

So although it looked more capable on the surface, it also showed clear hallucination and weak grounding.

#### 3) `Qwen/Qwen2.5-1.5B-Instruct`
This model had the highest overhead of the three, with a **53.75 second** load time and **0.81 second** generation time.

Its answer was the best of the test set:
- clearer structure,
- more coherent step-by-step explanation,
- more recognisable beginner-friendly style.

That said, it still included inaccuracies, such as generic scheduler advice and references like `qsub`, which are not aligned with the Bede Slurm workflow being evaluated.

So this model was the strongest in quality, but not fully reliable.

## 5. Interpretation

This test shows a clear trade-off between **speed** and **usefulness**.

The smallest model, `facebook/opt-125m`, was extremely fast, but its answer quality was too poor to be useful for HPC guidance. It may be acceptable only as a pipeline smoke test, not as a meaningful assistant for users.

The two Qwen instruct models produced much richer answers, which is a strong sign that instruction-tuned models are more appropriate for this kind of research-support or documentation task. However, even the stronger models still produced:
- generic HPC advice,
- mixed accuracy,
- and signs of hallucination or mismatch with the actual Bede workflow.

### Main takeaway
For this type of task:
- **very small models are fast but not dependable**
- **larger instruct models are more useful, but still need human review**
- **load time becomes a significant practical cost**, especially in short HPC jobs

### Practical conclusion for researchers
If the goal is:
- **quick environment validation**, a tiny model is enough
- **usable explanatory output**, a stronger instruct model is preferable
- **accurate operational guidance**, model output should still be checked against real cluster documentation and tested workflows

This is especially important in HPC settings, where small inaccuracies in job submission, scheduler commands, or container usage can cause jobs to fail or confuse new users.

### What this suggests for future testing
A stronger follow-up version of this experiment would:
- compare models that are closer in family but different in size,
- use warm caches to reduce first-run download effects,
- and include a small evaluation rubric for:
  - correctness,
  - clarity,
  - beginner usefulness,
  - and hallucination risk.